# Jamii Afya Falcon production pipeline

This notebook is audit-first and resume-first. It does not launch a long training run unless `FALCON_RUN_MODE=stage` is explicitly set. Every child command is streamed line-by-line, and the trainer writes JSONL events, metrics, heartbeats, and Trainer checkpoints.


In [ ]:
import json, os, selectors, shutil, subprocess, sys, time
from pathlib import Path
WORK = Path('/kaggle/working')
REPO = WORK / 'adtc-llm-limited-hardware'
EXP = REPO / 'experiments' / 'falcon-production-v1'
LOG = WORK / 'falcon-production-bootstrap-logs'
BRANCH = 'research/edge35-adaptive-streaming'
# Safe by default: audit only. Training modes must be explicitly selected.
RUN_MODE = os.environ.get('FALCON_RUN_MODE', 'stage')
STAGE = os.environ.get('FALCON_STAGE', 'stage_a_capability_preserving')
CONFIG = os.environ.get('FALCON_CONFIG', 'configs/falcon-production-v1.json')
SYSTEM_PROMPT_FILE = os.environ.get('FALCON_SYSTEM_PROMPT_FILE', '')
CHECKPOINT_DATASET = os.environ.get('FALCON_CHECKPOINT_DATASET', 'toheebogunade/jamii-afya-falcon-production-checkpoints')
MAX_STEPS = os.environ.get('FALCON_MAX_STEPS', '16')
SAVE_STEPS = os.environ.get('FALCON_SAVE_STEPS', '4')
EVAL_STEPS = os.environ.get('FALCON_EVAL_STEPS', '4')
os.environ.setdefault('FALCON_QUALITY_CANDIDATE_LIMIT', '1')
os.environ.setdefault('FALCON_MAX_DEV_EVAL', '8')
os.environ.setdefault('FALCON_EVAL_MAX_NEW_TOKENS', '96')
os.environ.setdefault('FALCON_STAGE_BATTERIES', 'docs/research/falcon_probe_heldout.json')
if RUN_MODE == 'pilot': os.environ.setdefault('FALCON_MCQA_MAX_PER_DATASET', '4')
INIT_ADAPTER = os.environ.get('FALCON_INIT_ADAPTER', '')
LOG.mkdir(parents=True, exist_ok=True)
def streamed(command, cwd=REPO, name='command.log', env=None):
    path = LOG / name
    merged = os.environ.copy(); merged.update(env or {}); merged['PYTHONUNBUFFERED'] = '1'
    print('STREAM', ' '.join(map(str, command)), flush=True)
    with path.open('a', encoding='utf-8', buffering=1) as handle:
        proc = subprocess.Popen(command, cwd=cwd, env=merged, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        assert proc.stdout is not None
        selector = selectors.DefaultSelector(); selector.register(proc.stdout, selectors.EVENT_READ)
        started = time.monotonic()
        try:
            while True:
                ready = selector.select(timeout=30)
                if ready:
                    line = proc.stdout.readline()
                    if line:
                        stamp = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())
                        rendered = f'[{stamp}] {line}'
                        print(rendered, end='', flush=True); handle.write(rendered); handle.flush()
                    elif proc.poll() is not None:
                        break
                else:
                    stamp = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())
                    heartbeat = f'[{stamp}] HEARTBEAT child={name} pid={proc.pid} elapsed={time.monotonic()-started:.1f}s\n'
                    print(heartbeat, end='', flush=True); handle.write(heartbeat); handle.flush()
                if proc.poll() is not None:
                    for line in proc.stdout:
                        stamp = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())
                        rendered = f'[{stamp}] {line}'
                        print(rendered, end='', flush=True); handle.write(rendered); handle.flush()
                    break
        finally:
            selector.close()
        code = proc.wait()
        print(f'EXIT={code}', flush=True); handle.write(f'EXIT={code}\n')
    if code: raise RuntimeError(f'command failed: {command}')
if not (REPO / '.git').exists():
    if REPO.exists(): shutil.rmtree(REPO)
    streamed(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/qeinstein/adtc-llm-limited-hardware.git',str(REPO)], cwd=WORK, name='clone.log')
else:
    # Kaggle may reuse a worker checkout; never trust its previous branch/SHA.
    streamed(['git','-C',str(REPO),'fetch','origin',BRANCH], cwd=WORK, name='git-refresh.log')
    streamed(['git','-C',str(REPO),'checkout','-B',BRANCH,'origin/'+BRANCH], cwd=WORK, name='git-refresh.log')
print('repo sha:', subprocess.run(['git','-C',str(REPO),'rev-parse','HEAD'], text=True, capture_output=True, check=True).stdout.strip(), flush=True)
if not (REPO / 'requirements-falcon-production.txt').exists(): raise RuntimeError('clean checkout is missing pinned production requirements')
LOG = EXP / 'kaggle-logs'; LOG.mkdir(parents=True, exist_ok=True)
streamed([sys.executable,'-m','pip','install','-q','-r','requirements-falcon-production.txt'], name='pip.log')
gpu = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'], text=True, capture_output=True, check=False).stdout.strip()
if 'P100' in gpu:
    # P100/sm_60 cannot use the image's current Torch or bitsandbytes path.
    streamed([sys.executable,'-m','pip','install','-q','--upgrade','numpy<2'], name='numpy-p100.log')
    # Torch >=2.6 is required by Transformers when restoring optimizer/scheduler
    # state. cu118 still supports the P100/sm_60 path used by this worker.
    streamed([sys.executable,'-m','pip','install','-q','--upgrade','--force-reinstall','torch==2.6.0','--index-url','https://download.pytorch.org/whl/cu118'], name='torch-p100.log')
    streamed([sys.executable,'-m','pip','uninstall','-y','torchao','torchvision','torchaudio','bitsandbytes'], name='optional-uninstall.log')
    streamed([sys.executable,'-m','pip','install','-q','--upgrade','--force-reinstall','--no-deps','transformers==4.53.3','tokenizers==0.21.4','peft==0.15.2','accelerate==1.7.0'], name='hf-stack-final.log')
else:
    streamed([sys.executable,'-m','pip','install','-q','bitsandbytes==0.50.2'], name='bitsandbytes.log')
streamed([sys.executable, '-c', 'import importlib.metadata as m, torch; print({"torch":torch.__version__,"cuda":torch.version.cuda, **{x:m.version(x) for x in ("transformers","peft","accelerate")}})'], name='environment-final.log')
print(json.dumps({'run_mode': RUN_MODE, 'stage': STAGE, 'gpu': gpu, 'checkpoint_dataset_configured': bool(CHECKPOINT_DATASET), 'repo': str(REPO)}, indent=2), flush=True)

In [ ]:
if RUN_MODE == 'generation_smoke':
    print('GENERATION_SMOKE: skip dataset build; holdout/data build is covered by audit versions.', flush=True)
else:
    DATA_DIR = EXP / 'data'
    if SYSTEM_PROMPT_FILE:
        prompt_path = REPO / SYSTEM_PROMPT_FILE
        if not prompt_path.is_file(): raise RuntimeError(f'system prompt file missing: {prompt_path}')
        prompt_payload = json.loads(prompt_path.read_text())
        if not prompt_payload.get('id') or not prompt_payload.get('text'): raise RuntimeError('selected system prompt must contain id and text')
        prompt_config = json.loads((REPO / CONFIG).read_text())
        prompt_config['data']['system_prompt'] = prompt_payload['text']
        prompt_config['data']['system_prompt_id'] = prompt_payload['id']
        prompt_config_path = WORK / 'falcon-selected-prompt-config.json'
        prompt_config_path.write_text(json.dumps(prompt_config, indent=2, ensure_ascii=False) + '\n')
        CONFIG = str(prompt_config_path)
        print(json.dumps({'selected_system_prompt_id':prompt_payload['id'],'prompt_config':CONFIG}, indent=2), flush=True)
    streamed([sys.executable, '-u', 'scripts/verify_falcon_holdouts.py'], name='holdout-freeze-check.log')
    if DATA_DIR.exists(): shutil.rmtree(DATA_DIR)
    streamed([sys.executable, '-u', 'scripts/audit_falcon_data.py', '--config', CONFIG, '--tokenizer', 'tiiuae/Falcon-H1-1.5B-Deep-Instruct', '--out', str(EXP / 'raw-data-audit-exact.json')], name='raw-data-audit.log')
    MCQA = REPO / 'output' / 'accuracy_sft.jsonl'
    if MCQA.exists() and os.environ.get('FALCON_REUSE_BUILT_DATA') != '1': MCQA.unlink()
    if not MCQA.exists():
        # Public TRAIN splits only; the cap is explicit so the first clean-worker
        # audit is bounded. Increase only after measured throughput/token-share evidence.
        mcqa_cfg = json.loads((REPO / CONFIG).read_text())['data']['mcqa']
        cap = os.environ.get('FALCON_MCQA_MAX_PER_DATASET', str(mcqa_cfg['max_per_dataset']))
        streamed([sys.executable, '-u', 'scripts/build_accuracy_sft.py', '--datasets', *mcqa_cfg['datasets'], '--max-per-dataset', cap, '--letter-permutations', str(mcqa_cfg['letter_permutations']), '--seed', str(mcqa_cfg['seed']), '--fail-on-source-error', '--out', str(MCQA)], name='mcqa-build.log')
    streamed([sys.executable, '-u', 'scripts/build_falcon_dataset.py', '--config', CONFIG, '--out-dir', str(DATA_DIR)], name='dataset-build.log')
    manifest = json.loads((DATA_DIR / 'data_manifest.json').read_text())
    print(json.dumps({k: manifest[k] for k in ('counts','token_totals','token_shares_percent','loss_token_totals','loss_token_shares_percent','facets','missing_sources')}, indent=2), flush=True)
    if RUN_MODE == 'audit':
        print('AUDIT_ONLY: no training launched. Set FALCON_RUN_MODE=resume_test or stage after persistence is configured.', flush=True)
        # Exercise the exact post-tokenization dataset path without loading
        # model weights. This catches objective-balance drift before GPU use.
        preflight_dir = EXP / 'preflight'
        streamed([sys.executable, '-u', 'scripts/train_falcon_production.py', '--config', CONFIG, '--data-dir', str(DATA_DIR), '--run-dir', str(preflight_dir), '--stage', STAGE, '--dry-run', '--allow-ephemeral'], name='tokenized-preflight.log')
        print('TOKENIZED_PREFLIGHT_PASS: exact tokenizer/data balance and truncation checks passed; no model weights loaded.', flush=True)


In [ ]:
if RUN_MODE == 'persistence_verify':
    # Re-check the already-uploaded checkpoint without repeating training.
    persist_dataset = CHECKPOINT_DATASET or 'toheebogunade/jamii-afya-falcon-production-checkpoints'
    verify_dir = EXP / 'resume-persistence-verification'
    if verify_dir.exists(): shutil.rmtree(verify_dir)
    streamed([sys.executable, '-u', 'scripts/verify_persisted_checkpoint.py', '--dataset', persist_dataset, '--out-dir', str(verify_dir)], env={'FALCON_CHECKPOINT_DATASET': persist_dataset}, name='persistence-only-verify.log')
    print('PERSISTENCE_ONLY_PASS: Kaggle checkpoint download contains resumable trainer state.', flush=True)
elif RUN_MODE == 'resume_test':
    # This is intentionally tiny and ephemeral: it verifies Trainer checkpoint
    # state/resume mechanics before any expensive production stage.
    test_run = EXP / 'resume-test'
    streamed([sys.executable, '-u', 'scripts/train_falcon_production.py', '--config', CONFIG, '--data-dir', str(DATA_DIR), '--run-dir', str(test_run), '--stage', STAGE, '--max-steps', '2', '--save-steps', '1', '--allow-ephemeral'], name='resume-test-first.log')
    streamed([sys.executable, '-u', 'scripts/train_falcon_production.py', '--config', CONFIG, '--data-dir', str(DATA_DIR), '--run-dir', str(test_run), '--stage', STAGE, '--max-steps', '4', '--save-steps', '1', '--resume-from-checkpoint', 'latest', '--allow-ephemeral'], name='resume-test-resume.log')
    ckpt_root = test_run / STAGE / 'checkpoints'
    first = json.loads((ckpt_root / 'checkpoint-2' / 'trainer_state.json').read_text())
    resumed = json.loads((ckpt_root / 'checkpoint-4' / 'trainer_state.json').read_text())
    assert first['global_step'] == 2 and resumed['global_step'] == 4
    assert any((ckpt_root / 'checkpoint-2').glob('optimizer.*')) and any((ckpt_root / 'checkpoint-2').glob('scheduler.*')) and (ckpt_root / 'checkpoint-2' / 'rng_state.pth').exists()
    persist_dataset = CHECKPOINT_DATASET or 'toheebogunade/jamii-afya-falcon-production-checkpoints'
    streamed([sys.executable, '-u', 'scripts/persist_checkpoint.py', '--checkpoint', str(ckpt_root / 'checkpoint-4'), '--dataset', persist_dataset, '--message', 'falcon-production-v1 resume-test persistence proof'], env={'FALCON_CHECKPOINT_DATASET': persist_dataset}, name='resume-test-persist.log')
    verify_dir = EXP / 'resume-persistence-verification'
    if verify_dir.exists(): shutil.rmtree(verify_dir)
    streamed([sys.executable, '-u', 'scripts/verify_persisted_checkpoint.py', '--dataset', persist_dataset, '--out-dir', str(verify_dir)], env={'FALCON_CHECKPOINT_DATASET': persist_dataset}, name='resume-test-verify.log')
    print('RESUME_TEST_PASS: global_step 2 -> 4, optimizer/scheduler/RNG state present, and checkpoint persisted/retrieved from Kaggle Dataset.', flush=True)


In [ ]:
def choose_persisted_checkpoint(dataset, purpose, requested_step=None):
    destination = EXP / purpose
    if destination.exists(): shutil.rmtree(destination)
    streamed([sys.executable, '-u', 'scripts/verify_persisted_checkpoint.py', '--dataset', dataset, '--out-dir', str(destination)], env={'FALCON_CHECKPOINT_DATASET': dataset}, name=f'{purpose}.log')
    candidates = []
    for state_path in destination.rglob('trainer_state.json'):
        state = json.loads(state_path.read_text())
        if isinstance(state.get('global_step'), int): candidates.append((state['global_step'], state_path.parent))
    if not candidates: raise RuntimeError(f'no resumable checkpoint found in {dataset}')
    if requested_step is not None:
        exact = [item for item in candidates if item[0] == int(requested_step)]
        if not exact: raise RuntimeError(f'checkpoint step {requested_step} not found in persisted dataset')
        selected = exact[0]
    else: selected = max(candidates, key=lambda item: item[0])
    print(json.dumps({'purpose': purpose, 'checkpoint': str(selected[1]), 'global_step': selected[0]}), flush=True)
    return selected[1]
if RUN_MODE == 'persisted_eval':
    eval_dataset = CHECKPOINT_DATASET or 'toheebogunade/jamii-afya-falcon-production-checkpoints'
    eval_step = os.environ.get('FALCON_EVAL_STEP')
    checkpoint = choose_persisted_checkpoint(eval_dataset, 'persisted-eval-checkpoint', eval_step)
    eval_id = os.environ.get('FALCON_EVAL_ID') or time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())
    eval_dir = EXP / 'persisted-evals' / eval_id
    eval_cmd = [sys.executable, '-u', 'scripts/evaluate_falcon_hf.py', '--config', CONFIG, '--data-dir', str(DATA_DIR), '--output-dir', str(eval_dir), '--adapter', str(checkpoint), '--max-dev', os.environ.get('FALCON_MAX_DEV_EVAL', '64'), '--max-new-tokens', os.environ.get('FALCON_EVAL_MAX_NEW_TOKENS', '128'), '--battery', 'docs/research/falcon_probe_heldout.json']
    streamed(eval_cmd, name='persisted-eval.log')
    streamed([sys.executable, '-u', 'scripts/score_falcon_battery.py', '--battery', 'docs/research/falcon_probe_heldout.json', '--generation-dir', str(eval_dir / 'falcon_probe_heldout'), '--out', str(eval_dir / 'generation_quality.json'), '--report-only'], name='persisted-eval-quality.log')
    print(json.dumps({'PERSISTED_EVAL_COMPLETE': True, 'checkpoint': str(checkpoint), 'eval_dir': str(eval_dir)}, indent=2), flush=True)
elif RUN_MODE == 'persisted_compare':
    eval_dataset = CHECKPOINT_DATASET or 'toheebogunade/jamii-afya-falcon-production-checkpoints'
    checkpoint = choose_persisted_checkpoint(eval_dataset, 'persisted-compare-checkpoint', os.environ.get('FALCON_EVAL_STEP'))
    compare_id = os.environ.get('FALCON_EVAL_ID') or time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())
    compare_root = EXP / 'persisted-compares' / compare_id
    for label, adapter in [('stock', None), ('step8_adapter', checkpoint)]:
        output_dir = compare_root / label
        eval_cmd = [sys.executable, '-u', 'scripts/evaluate_falcon_hf.py', '--config', CONFIG, '--data-dir', str(DATA_DIR), '--output-dir', str(output_dir), '--max-dev', os.environ.get('FALCON_MAX_DEV_EVAL', '64'), '--max-new-tokens', os.environ.get('FALCON_EVAL_MAX_NEW_TOKENS', '64'), '--battery', 'docs/research/falcon_probe_heldout.json']
        if adapter is not None: eval_cmd += ['--adapter', str(adapter)]
        streamed(eval_cmd, name=f'persisted-compare-{label}.log')
        streamed([sys.executable, '-u', 'scripts/score_falcon_battery.py', '--battery', 'docs/research/falcon_probe_heldout.json', '--generation-dir', str(output_dir / 'falcon_probe_heldout'), '--out', str(output_dir / 'generation_quality.json'), '--report-only'], name=f'persisted-compare-{label}-quality.log')
    print(json.dumps({'PERSISTED_COMPARE_COMPLETE': True, 'checkpoint': str(checkpoint), 'compare_root': str(compare_root)}, indent=2), flush=True)
elif RUN_MODE == 'generation_smoke':
    smoke_dir = EXP / 'generation-smoke' / (os.environ.get('FALCON_SMOKE_ID') or time.strftime('%Y%m%dT%H%M%SZ', time.gmtime()))
    smoke_dir.mkdir(parents=True, exist_ok=True)
    streamed([sys.executable, '-u', 'scripts/falcon_hf_generation_smoke.py', '--model', 'tiiuae/Falcon-H1-1.5B-Deep-Instruct', '--revision', json.loads((REPO / CONFIG).read_text())['model']['revision'], '--output', str(smoke_dir / 'smoke.json'), '--max-new-tokens', os.environ.get('FALCON_SMOKE_MAX_NEW_TOKENS', '64')], name='generation-smoke.log')
    print(json.dumps({'GENERATION_SMOKE_COMPLETE': True, 'output': str(smoke_dir / 'smoke.json')}, indent=2), flush=True)
elif RUN_MODE == 'export':
    # Export is deliberately a separate Kaggle mode: the base model, merged
    # HF weights, F16 GGUF, and quantized GGUF never touch this workstation.
    adapter = Path(os.environ.get('FALCON_ADAPTER_PATH', '')).resolve()
    if not adapter.is_dir(): raise RuntimeError('FALCON_ADAPTER_PATH must point to a retrieved promoted adapter directory')
    from scripts.verify_falcon_promotion import verify_frozen_quality_report, verify_promoted_adapter
    promotion_manifest = Path(os.environ.get('FALCON_PROMOTION_MANIFEST', str(adapter / 'promotion_manifest.json'))).resolve()
    promotion = verify_promoted_adapter(adapter, manifest_path=promotion_manifest, minimum_pass_rate=100.0)
    print(json.dumps({'PROMOTION_INPUT_VERIFIED': True, **promotion}, indent=2), flush=True)
    export_dir = EXP / 'export' / (os.environ.get('FALCON_EXPORT_ID') or time.strftime('%Y%m%dT%H%M%SZ', time.gmtime()))
    export_env = {'BASE_MODEL': 'tiiuae/Falcon-H1-1.5B-Deep-Instruct', 'MODEL_REVISION': json.loads((REPO / CONFIG).read_text())['model']['revision'], 'PROMOTION_MANIFEST': str(promotion_manifest)}
    streamed(['bash', 'scripts/export_falcon_gguf.sh', str(adapter), str(export_dir)], env=export_env, name='export.log')
    merged = export_dir / 'merged-hf'
    fp16_eval = export_dir / 'merged-eval'
    eval_cmd = [sys.executable, '-u', 'scripts/evaluate_falcon_hf.py', '--config', CONFIG, '--data-dir', str(DATA_DIR), '--output-dir', str(fp16_eval), '--merged-model', str(merged), '--max-dev', os.environ.get('FALCON_MAX_DEV_EVAL', '200'), '--max-new-tokens', os.environ.get('FALCON_EVAL_MAX_NEW_TOKENS', '256')]
    for battery in os.environ.get('FALCON_STAGE_BATTERIES', 'docs/research/falcon_probe_heldout.json,docs/research/falcon_baseline_prompts.json,data/swahili_eval_set.json').split(','):
        if battery.strip(): eval_cmd += ['--battery', battery.strip()]
    streamed(eval_cmd, name='merged-eval.log')
    frozen_battery = 'docs/research/falcon_probe_heldout.json'
    export_batteries = os.environ.get('FALCON_STAGE_BATTERIES', 'docs/research/falcon_probe_heldout.json,docs/research/falcon_baseline_prompts.json,data/swahili_eval_set.json').split(',')
    if frozen_battery not in {item.strip() for item in export_batteries}: raise RuntimeError('export must include the frozen clinical/safety battery')
    merged_quality_path = fp16_eval / 'generation_quality.json'
    streamed([sys.executable, '-u', 'scripts/score_falcon_battery.py', '--battery', frozen_battery, '--generation-dir', str(fp16_eval / 'falcon_probe_heldout'), '--out', str(merged_quality_path), '--report-only'], name='merged-quality-gate.log')
    merged_quality = verify_frozen_quality_report(merged_quality_path, minimum_pass_rate=100.0, expected_battery=frozen_battery)
    export_manifest = json.loads((export_dir / 'export_manifest.json').read_text())
    deployment = Path(export_manifest['deployment_model']).resolve()
    if not deployment.is_file(): raise RuntimeError(f'export manifest points to missing deployment model: {deployment}')
    candidate_dir = export_dir / 'quantized-eval'
    candidate_cmd = [sys.executable, '-u', 'scripts/evaluate_falcon_candidate.py', '--model', str(deployment), '--out-dir', str(candidate_dir), '--limit', os.environ.get('FALCON_FINAL_MCQA_LIMIT', '500'), '--n-ctx', '2048', '--generation-mode', 'chat', '--system-prompt', json.loads((REPO / CONFIG).read_text())['data']['system_prompt']]
    for battery in os.environ.get('FALCON_STAGE_BATTERIES', 'docs/research/falcon_probe_heldout.json,docs/research/falcon_baseline_prompts.json,data/swahili_eval_set.json').split(','):
        if battery.strip(): candidate_cmd += ['--battery', battery.strip()]
    streamed(candidate_cmd, name='quantized-eval.log')
    quantized_quality_path = candidate_dir / 'generation_quality.json'
    streamed([sys.executable, '-u', 'scripts/score_falcon_battery.py', '--battery', frozen_battery, '--generation-dir', str(candidate_dir / 'falcon_probe_heldout'), '--out', str(quantized_quality_path), '--report-only'], name='quantized-quality-gate.log')
    quantized_quality = verify_frozen_quality_report(quantized_quality_path, minimum_pass_rate=100.0, expected_battery=frozen_battery)
    export_summary = {'status': 'exported_and_frozen_gate_passed', 'export_dir': str(export_dir), 'promotion': promotion, 'merged_eval': str(fp16_eval), 'quantized_eval': str(candidate_dir), 'merged_frozen_quality': {'pass_rate_percent': merged_quality.get('pass_rate_percent'), 'critical_failures': merged_quality.get('critical_failures', [])}, 'quantized_frozen_quality': {'pass_rate_percent': quantized_quality.get('pass_rate_percent'), 'critical_failures': quantized_quality.get('critical_failures', [])}}
    (export_dir / 'export_quality_summary.json').write_text(json.dumps(export_summary, indent=2) + '\n')
    print(json.dumps({'EXPORT_COMPLETE': True, **export_summary}, indent=2), flush=True)
elif RUN_MODE == 'pilot':
    # Bounded, persisted Stage-A experiment. This is the first meaningful
    # quality run after the infrastructure gates; it is not the final run.
    pilot_dataset = CHECKPOINT_DATASET or 'toheebogunade/jamii-afya-falcon-production-checkpoints'
    pilot_id = os.environ.get('FALCON_PILOT_ID') or 'falcon-small-real-probe-v1-20260913'
    run_dir = EXP / 'pilots' / pilot_id
    pilot_steps = os.environ.get('FALCON_PILOT_STEPS', '8')
    pilot_profile = os.environ.get('FALCON_PILOT_PROFILE', 'safe_attention')
    pilot_eval_max_tokens = os.environ.get('FALCON_PILOT_MAX_NEW_TOKENS', '128')
    pilot_batteries = os.environ.get('FALCON_PILOT_BATTERIES', 'docs/research/falcon_prompt_dev.json,docs/research/falcon_prompt_validation.json').split(',')
    env = {'FALCON_CHECKPOINT_DATASET': pilot_dataset}
    command = [sys.executable, '-u', 'scripts/train_falcon_production.py', '--config', CONFIG, '--data-dir', str(DATA_DIR), '--run-dir', str(run_dir), '--stage', STAGE, '--max-steps', pilot_steps, '--save-steps', '4', '--eval-steps', os.environ.get('FALCON_PILOT_EVAL_STEPS', '4')]
    if pilot_profile == 'safe_attention': command += ['--learning-rate', '0.000005', '--lora-r', '8', '--lora-alpha', '16', '--lora-dropout', '0.1', '--target-modules', 'q_proj,k_proj,v_proj,o_proj']
    elif pilot_profile != 'canonical': raise RuntimeError(f'unknown FALCON_PILOT_PROFILE: {pilot_profile}')
    streamed(command, env=env, name=f'pilot-{STAGE}.log')
    adapter = run_dir / STAGE / 'checkpoints' / 'final-adapter'
    if not adapter.is_dir(): raise RuntimeError(f'pilot did not produce final adapter: {adapter}')
    baseline_dir = run_dir / STAGE / 'pilot-baseline'
    baseline_cmd = [sys.executable, '-u', 'scripts/evaluate_falcon_hf.py', '--config', CONFIG, '--data-dir', str(DATA_DIR), '--output-dir', str(baseline_dir), '--max-dev', os.environ.get('FALCON_MAX_DEV_EVAL', '64'), '--max-new-tokens', pilot_eval_max_tokens]
    for battery in pilot_batteries:
        if battery.strip(): baseline_cmd += ['--battery', battery.strip()]
    streamed(baseline_cmd, name='pilot-baseline-eval.log')
    eval_dir = run_dir / STAGE / 'pilot-eval'
    eval_cmd = [sys.executable, '-u', 'scripts/evaluate_falcon_hf.py', '--config', CONFIG, '--data-dir', str(DATA_DIR), '--output-dir', str(eval_dir), '--adapter', str(adapter), '--max-dev', os.environ.get('FALCON_MAX_DEV_EVAL', '64'), '--max-new-tokens', pilot_eval_max_tokens]
    for battery in pilot_batteries:
        if battery.strip(): eval_cmd += ['--battery', battery.strip()]
    streamed(eval_cmd, name='pilot-eval.log')
    for battery in pilot_batteries:
        battery = battery.strip()
        if not battery: continue
        battery_path = Path(battery)
        quality_out = eval_dir / f'{battery_path.stem}-quality.json'
        streamed([sys.executable, '-u', 'scripts/score_falcon_battery.py', '--battery', battery, '--generation-dir', str(eval_dir / battery_path.stem), '--out', str(quality_out), '--report-only'], name=f'pilot-eval-quality-{battery_path.stem}.log')
    print('FINAL_FROZEN_GATE_DEFERRED: no prompt is selected; pilot uses development/validation batteries only.', flush=True)
    print(json.dumps({'PILOT_COMPLETE': True, 'pilot_id': pilot_id, 'profile': pilot_profile, 'steps': int(pilot_steps), 'baseline_eval': str(baseline_dir), 'adapter': str(adapter), 'eval': str(eval_dir)}, indent=2), flush=True)
elif RUN_MODE == 'stage':
    if not CHECKPOINT_DATASET:
        raise RuntimeError('Set FALCON_CHECKPOINT_DATASET to an existing private Kaggle dataset before a production stage.')
    streamed(['kaggle', 'datasets', 'files', '-d', CHECKPOINT_DATASET], cwd=REPO, name='checkpoint-dataset-preflight.log')
    run_dir = EXP / 'runs' / (os.environ.get('FALCON_RUN_ID') or time.strftime('%Y%m%dT%H%M%SZ', time.gmtime()))
    requested_step = os.environ.get('FALCON_INIT_CHECKPOINT_STEP')
    init_adapter = Path(INIT_ADAPTER) if INIT_ADAPTER else None
    resume_arg = os.environ.get('FALCON_RESUME')
    if STAGE != 'stage_a_capability_preserving' and init_adapter is None:
        init_adapter = choose_persisted_checkpoint(CHECKPOINT_DATASET, 'stage-init-checkpoint', requested_step)
    if resume_arg and (resume_arg == 'latest' or not Path(resume_arg).exists()):
        resume_dir = choose_persisted_checkpoint(CHECKPOINT_DATASET, 'resume-checkpoint', os.environ.get('FALCON_RESUME_STEP'))
        resume_arg = str(resume_dir)
    env = {'FALCON_CHECKPOINT_DATASET': CHECKPOINT_DATASET}
    command = [sys.executable, '-u', 'scripts/train_falcon_production.py', '--config', CONFIG, '--data-dir', str(DATA_DIR), '--run-dir', str(run_dir), '--stage', STAGE, '--max-steps', MAX_STEPS, '--save-steps', SAVE_STEPS, '--eval-steps', EVAL_STEPS, '--max-length', '384', '--lora-r', '4']
    if resume_arg:
        command += ['--resume-from-checkpoint', resume_arg]
    if init_adapter:
        command += ['--init-adapter', str(init_adapter)]
    streamed(command, env=env, name=f'{STAGE}.log')
    stage_dir = run_dir / STAGE
    selection_path = stage_dir / 'checkpoint_selection.json'
    if not selection_path.exists(): raise RuntimeError('stage produced no checkpoint selection artifact')
    selection = json.loads(selection_path.read_text())
    if selection.get('status') != 'selected': raise RuntimeError('stage had no held-out evaluation checkpoint; refusing to promote last checkpoint')
    from scripts.select_falcon_candidate import frozen_gate_passes, select_dev_validation_candidate
    from scripts.verify_falcon_promotion import build_promotion_manifest, sha256_file, verify_promoted_adapter
    batteries = [item.strip() for item in os.environ.get('FALCON_STAGE_BATTERIES', 'docs/research/falcon_probe_heldout.json,docs/research/falcon_baseline_prompts.json,data/swahili_eval_set.json').split(',') if item.strip()]
    frozen_battery = 'docs/research/falcon_probe_heldout.json'
    dev_battery = 'docs/research/falcon_prompt_dev.json'
    validation_battery = 'docs/research/falcon_prompt_validation.json'
    if frozen_battery not in batteries: raise RuntimeError('production stage must include the frozen clinical/safety battery; refusing to persist an unevaluated checkpoint')
    for required_battery in (dev_battery, validation_battery):
        if required_battery not in batteries: batteries.append(required_battery)
    quality_root = run_dir / STAGE / 'checkpoint-quality'
    quality_root.mkdir(parents=True, exist_ok=True)
    quality_candidates = sorted(selection.get('candidates', []), key=lambda item: (float(item.get('eval_loss', 1e30)), int(item.get('step', 0))))
    quality_limit = int(os.environ.get('FALCON_QUALITY_CANDIDATE_LIMIT', '3'))
    dev_validation_reports = {}
    for candidate in quality_candidates[:max(1, quality_limit)]:
        candidate_path = Path(candidate['checkpoint'])
        if not candidate_path.is_dir(): continue
        step = int(candidate['step'])
        eval_dir = quality_root / f'step-{step}'
        dev_validation_reports[step] = {}
        eval_cmd = [sys.executable, '-u', 'scripts/evaluate_falcon_hf.py', '--config', CONFIG, '--data-dir', str(DATA_DIR), '--output-dir', str(eval_dir), '--adapter', str(candidate_path), '--max-dev', os.environ.get('FALCON_MAX_DEV_EVAL', '64'), '--max-new-tokens', os.environ.get('FALCON_EVAL_MAX_NEW_TOKENS', '128'), '--battery', dev_battery, '--battery', validation_battery]
        for battery in batteries:
            if battery not in {frozen_battery, dev_battery, validation_battery}: eval_cmd += ['--battery', battery]
        try:
            streamed(eval_cmd, name=f'{STAGE}-quality-eval-step-{step}.log')
            for label, battery in (('dev', dev_battery), ('validation', validation_battery)):
                battery_stem = Path(battery).stem
                quality_path = eval_dir / f'{label}-quality.json'
                streamed([sys.executable, '-u', 'scripts/score_falcon_battery.py', '--battery', battery, '--generation-dir', str(eval_dir / battery_stem), '--out', str(quality_path), '--report-only'], name=f'{STAGE}-quality-{label}-step-{step}.log')
                dev_validation_reports[step][label] = json.loads(quality_path.read_text())
        except (OSError, ValueError, RuntimeError) as exc:
            print(json.dumps({'quality_candidate_step': step, 'status': 'evaluation_failed', 'error': str(exc)}), flush=True)
    minimum_dev_validation_pass_rate = float(os.environ.get('FALCON_MIN_DEV_VALIDATION_PASS_RATE', '75'))
    quality_selection = select_dev_validation_candidate(quality_candidates[:max(1, quality_limit)], dev_validation_reports, minimum_pass_rate=minimum_dev_validation_pass_rate)
    quality_selection['frozen_gate'] = {'status': 'deferred_until_dev_validation_selection', 'battery': frozen_battery}
    (run_dir / STAGE / 'quality_selection.json').write_text(json.dumps(quality_selection, indent=2) + '\n')
    if quality_selection.get('status') != 'selected_for_frozen_gate': raise RuntimeError('no loss-ranked checkpoint passed the development/validation quality gate; refusing to read frozen holdout or persist')
    adapter = Path(quality_selection['selected_checkpoint'])
    selected_step = int(quality_selection['selected_step'])
    frozen_dir = quality_root / f'selected-step-{selected_step}-frozen'
    frozen_eval_cmd = [sys.executable, '-u', 'scripts/evaluate_falcon_hf.py', '--config', CONFIG, '--data-dir', str(DATA_DIR), '--output-dir', str(frozen_dir), '--adapter', str(adapter), '--max-dev', os.environ.get('FALCON_MAX_DEV_EVAL', '64'), '--max-new-tokens', os.environ.get('FALCON_EVAL_MAX_NEW_TOKENS', '128'), '--battery', frozen_battery]
    streamed(frozen_eval_cmd, name=f'{STAGE}-frozen-eval-step-{selected_step}.log')
    frozen_quality_path = frozen_dir / 'frozen-quality.json'
    streamed([sys.executable, '-u', 'scripts/score_falcon_battery.py', '--battery', frozen_battery, '--generation-dir', str(frozen_dir / Path(frozen_battery).stem), '--out', str(frozen_quality_path), '--report-only'], name=f'{STAGE}-frozen-gate-step-{selected_step}.log')
    frozen_summary = json.loads(frozen_quality_path.read_text())
    frozen_pass = frozen_gate_passes(frozen_summary, float(os.environ.get('FALCON_MIN_FROZEN_PASS_RATE', '100')))
    quality_selection['frozen_gate'] = {'status': 'passed' if frozen_pass else 'rejected', 'battery': frozen_battery, 'report': str(frozen_quality_path), 'pass_rate_percent': frozen_summary.get('pass_rate_percent', 0.0), 'critical_failures': frozen_summary.get('critical_failures', []), 'minimum_pass_rate_percent': float(os.environ.get('FALCON_MIN_FROZEN_PASS_RATE', '100'))}
    quality_selection['status'] = 'selected_and_frozen_gate_passed' if frozen_pass else 'selected_but_frozen_gate_rejected'
    (run_dir / STAGE / 'quality_selection.json').write_text(json.dumps(quality_selection, indent=2) + '\n')
    if not frozen_pass: raise RuntimeError('selected checkpoint failed the frozen clinical/safety quality gate; refusing to persist')
    repo_sha = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True, capture_output=True, check=True).stdout.strip()
    config_path = Path(CONFIG) if Path(CONFIG).is_absolute() else REPO / CONFIG
    data_manifest_path = DATA_DIR / 'data_manifest.json'
    promotion_manifest = build_promotion_manifest(adapter, quality_selection=quality_selection, frozen_report={**frozen_summary, 'report_sha256': sha256_file(frozen_quality_path)}, experiment_id=json.loads(config_path.read_text())['experiment_id'], stage=STAGE, repo_sha=repo_sha, config_sha256=sha256_file(config_path), data_manifest_sha256=sha256_file(data_manifest_path), minimum_pass_rate=float(os.environ.get('FALCON_MIN_FROZEN_PASS_RATE', '100')))
    promotion_path = adapter / 'promotion_manifest.json'
    promotion_path.write_text(json.dumps(promotion_manifest, indent=2, ensure_ascii=False) + '\n')
    checkpoint_manifest_path = adapter / 'checkpoint_manifest.json'
    checkpoint_manifest = json.loads(checkpoint_manifest_path.read_text())
    checkpoint_manifest['files'] = sorted(str(path.relative_to(adapter)) for path in adapter.rglob('*') if path.is_file())
    checkpoint_manifest['promotion_status'] = 'promoted_after_frozen_gate'
    checkpoint_manifest_path.write_text(json.dumps(checkpoint_manifest, indent=2) + '\n')
    final_adapter = stage_dir / 'checkpoints' / 'final-adapter'
    if not final_adapter.is_dir(): raise RuntimeError(f'materialized final adapter missing: {final_adapter}')
    final_promotion_path = final_adapter / 'promotion_manifest.json'
    final_promotion_path.write_text(json.dumps(promotion_manifest, indent=2, ensure_ascii=False) + '\n')
    verify_promoted_adapter(adapter, minimum_pass_rate=100.0)
    verify_promoted_adapter(final_adapter, minimum_pass_rate=100.0)
    final_summary_path = stage_dir / 'final_summary.json'
    final_summary = json.loads(final_summary_path.read_text())
    final_summary.update({'status': 'promoted', 'promotion_status': 'promoted_after_frozen_gate', 'promotion_manifest': str(final_promotion_path), 'frozen_quality_report': str(frozen_quality_path)})
    final_summary_path.write_text(json.dumps(final_summary, indent=2) + '\n')
    streamed([sys.executable, '-u', 'scripts/persist_checkpoint.py', '--checkpoint', str(adapter), '--dataset', CHECKPOINT_DATASET, '--message', f'{CONFIG} {STAGE} quality-gated step {selected_step}'], env=env, name=f'{STAGE}-selected-persist.log')
else:
    print('No training stage requested.', flush=True)